# Coding Assignment 1: Household support (implement, inspect, verify)

Research Methods 2026. Run with **Kernel → Restart Kernel and Run All Cells**. All results below come from the code in this notebook. Amounts are EUR per month; household size is persons.

In [1]:
import math
import sys

print("Python", sys.version)

Python 3.12.16 | packaged by conda-forge | (main, Oct 26 2026, 11:29:01) [MSC v.1944 64 bit (AMD64)]


## Inputs (supplied lists, kept unchanged)

In [2]:
household_ids = ["A01", "A02", "A03", "A04",
                 "A05", "A06", "A07", "A08"]
income_text = ["1200", "0", "1500", "700",
               "2400", "800", "0", "1800"]
consumptions = [1250, 150, 1600, 850, 2600, 950, 0, 1400]
household_sizes = [2, 1, 3, 1, 4, 2, 2, 3]

# Independent copies, used only to verify at the end that the supplied lists did not change.
original_ids = household_ids.copy()
original_income_text = income_text.copy()
original_consumptions = consumptions.copy()
original_sizes = household_sizes.copy()

## Question 1

### First prediction

- Line 2 does not copy the list. `saved_incomes` is a second name for the same list object as `incomes`.
- Line 3: `first_share = (1000 - 800) / 1000 = 0.2`. It is a float that is calculated once, now.
- Line 4 changes the shared list to `[1200, 800, 0]`.
- Line 5 appends 600 to the same list, so `incomes` is `[1200, 800, 0, 600]`.

Predicted output:
```
[1200, 800, 0, 600]
600 4
20%
```

In [3]:
incomes = [1000, 800, 0]
saved_incomes = incomes
first_share = (incomes[0] - 800) / incomes[0]
incomes[0] = 1200
saved_incomes.append(600)
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0, 600]
600 4
20%


### 1(a) Output and trace

The output agrees with my prediction, so no correction is necessary.

| Line | Effect | `incomes` (= `saved_incomes`) | `first_share` |
|---|---|---|---|
| 1 | Create one list object | `[1000, 800, 0]` | not defined |
| 2 | Bind a second name to the same object | `[1000, 800, 0]` | not defined |
| 3 | Calculate (1000 − 800) / 1000 | `[1000, 800, 0]` | `0.2` |
| 4 | Replace element 0 in the shared object | `[1200, 800, 0]` | `0.2` |
| 5 | Append 600 to the shared object | `[1200, 800, 0, 600]` | `0.2` |

Output lines: `[1200, 800, 0, 600]`, then `600 4` (last element 600; length 4), then `20%` (0.2 formatted as a percentage with no decimals).

### 1(b) Why

- **Append:** assignment (`saved_incomes = incomes`) copies the reference, not the list. Both names refer to one list object. `append` changes that object, so the change is visible through both names.
- **`first_share`:** line 3 calculates a number and stores the result (0.2). It does not store the formula. Python does not recalculate it when `incomes[0]` changes later. To get a new value, run the calculation again after the change.

### 1(c) Independent copy and recalculated share

Changes: line 2 uses `.copy()`; the share calculation now comes **after** the income change.

In [4]:
incomes = [1000, 800, 0]
saved_incomes = incomes.copy()                  # independent copy of the original list
incomes[0] = 1200
first_share = (incomes[0] - 800) / incomes[0]   # recalculated after the change
saved_incomes.append(600)                       # changes only the copy
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0]
600 3
33%


New output: `[1200, 800, 0]` (the append does not change `incomes`), `600 3` (the copy ends with 600; `incomes` has 3 elements), `33%` ((1200 − 800) / 1200 = 0.333...).

## Question 2. Implement the policy as functions

**Design (≤ 100 words).** Parameters are the names in each `def` line (`income`, `consumption`, `size`). Arguments are the values given in a call, for example `transfer(1200, 1250, 2)`. Each function uses only its parameters, so its result does not depend on values that earlier cells stored. `return` gives the value back to the caller; `print` only shows it, and the caller then gets `None`. `post_policy_share` needs the returned values to continue. The input checks come before any division, so invalid data stops with a clear `ValueError` and does not cause `ZeroDivisionError` or a meaningless ratio.

In [5]:
def saving_share(income, consumption):
    """Return (income - consumption) / income as a fraction, or None for zero income."""
    # Check invalid amounts first (before the zero-income case and before division).
    if income < 0 or consumption < 0:
        raise ValueError("income and consumption must be non-negative")
    if income == 0:
        return None          # share is undefined, not zero
    return (income - consumption) / income


def transfer(income, consumption, size):
    """Return the monthly transfer T(I, C, n) in EUR. Always a number for valid input."""
    # Validate all inputs before the eligibility test (which divides by size).
    if income < 0:
        raise ValueError("income must be non-negative")
    if consumption < 0:
        raise ValueError("consumption must be non-negative")
    if size <= 0:
        raise ValueError("size must be a positive integer")
    if income / size > 600:                 # above EUR 600 per person: not eligible
        return 0
    shortfall = max(consumption - income, 0)   # zero when there is no shortfall
    return min(100 * size, shortfall)          # cap at EUR 100 per person


def post_policy_share(income, consumption, size):
    """Saving share after support. Reuses transfer() and saving_share(); consumption is fixed."""
    support = transfer(income, consumption, size)
    return saving_share(income + support, consumption)

In [6]:
# Quick visible checks of the three functions (full tests are in Question 5).
print(saving_share(1000, 800))           # 0.2
print(saving_share(0, 150))              # None
print(transfer(1200, 1250, 2))           # 50: exactly EUR 600 per person is eligible
print(transfer(0, 150, 1))               # 100: shortfall 150 capped at 100 * 1
print(transfer(700, 850, 1))             # 0: EUR 700 per person is above the limit
print(post_policy_share(1200, 1250, 2))  # (1250 - 1250) / 1250 = 0.0

0.2
None
50
100
0
0.0


## Question 3. Build and interpret a checked report

### 3(a) Prepare inputs
The length check comes first. A mismatch raises `ValueError`, so `zip` cannot silently drop a household. The conversion makes a new list; `income_text` keeps its strings.

In [7]:
n_households = len(household_ids)
for name, values in [("income_text", income_text),
                     ("consumptions", consumptions),
                     ("household_sizes", household_sizes)]:
    if len(values) != n_households:
        raise ValueError(f"{name} has {len(values)} entries; household_ids has {n_households}")

income_numbers = []
for text in income_text:
    income_numbers.append(float(text))

print("Households:", n_households)
print("income_text (unchanged):", income_text)
print("income_numbers:", income_numbers)

Households: 8
income_text (unchanged): ['1200', '0', '1500', '700', '2400', '800', '0', '1800']
income_numbers: [1200.0, 0.0, 1500.0, 700.0, 2400.0, 800.0, 0.0, 1800.0]


### 3(b) Household results
Full precision is kept in the lists; rounding happens only in the printout. An undefined share prints as `undefined` and stays `None` in the list.

In [8]:
transfers = []
incomes_after = []
savings_before = []
savings_after = []
shares_before = []
shares_after = []

for hid, income, consumption, size in zip(household_ids, income_numbers,
                                          consumptions, household_sizes):
    t = transfer(income, consumption, size)
    y = income + t
    transfers.append(t)
    incomes_after.append(y)
    savings_before.append(income - consumption)
    savings_after.append(y - consumption)
    shares_before.append(saving_share(income, consumption))
    shares_after.append(post_policy_share(income, consumption, size))


def show_share(share):
    """Display a share as a percentage with 2 decimals; None is shown as 'undefined'."""
    if share is None:
        return "undefined"
    return f"{share:.2%}"


print("Household report (EUR per month; shares in % of the corresponding income)")
header = (f"{'ID':<4}{'Income':>10}{'Consump.':>10}{'Size':>5}{'Transfer':>10}"
          f"{'Sav. bef.':>11}{'Sav. aft.':>11}{'Share bef.':>12}{'Share aft.':>12}")
print(header)
print("-" * len(header))
for i in range(n_households):
    print(f"{household_ids[i]:<4}{income_numbers[i]:>10.2f}{consumptions[i]:>10.2f}"
          f"{household_sizes[i]:>5}{transfers[i]:>10.2f}{savings_before[i]:>11.2f}"
          f"{savings_after[i]:>11.2f}{show_share(shares_before[i]):>12}"
          f"{show_share(shares_after[i]):>12}")

Household report (EUR per month; shares in % of the corresponding income)
ID      Income  Consump. Size  Transfer  Sav. bef.  Sav. aft.  Share bef.  Share aft.
-------------------------------------------------------------------------------------
A01    1200.00   1250.00    2     50.00     -50.00       0.00      -4.17%       0.00%
A02       0.00    150.00    1    100.00    -150.00     -50.00   undefined     -50.00%
A03    1500.00   1600.00    3    100.00    -100.00       0.00      -6.67%       0.00%
A04     700.00    850.00    1      0.00    -150.00    -150.00     -21.43%     -21.43%
A05    2400.00   2600.00    4    200.00    -200.00       0.00      -8.33%       0.00%
A06     800.00    950.00    2    150.00    -150.00       0.00     -18.75%       0.00%
A07       0.00      0.00    2      0.00       0.00       0.00   undefined   undefined
A08    1800.00   1400.00    3      0.00     400.00     400.00      22.22%      22.22%


Saving amount and saving share are different quantities. Example: A01's saving amount rises by EUR 50 (from −50 to 0), and its share rises from −4.17% to 0.00%. A04 gets no transfer, so its amount (−150) and its share (−21.43%) do not change. A07 has zero income before and after support, so both of its shares stay undefined.

### 3(c) Totals and averages

In [9]:
total_transfer = 0
n_positive_transfer = 0
n_negative_before = 0
n_negative_after = 0
sum_saving_before = 0
sum_income_before = 0
sum_saving_after = 0
sum_income_after = 0
for i in range(n_households):
    total_transfer = total_transfer + transfers[i]
    if transfers[i] > 0:
        n_positive_transfer = n_positive_transfer + 1
    if savings_before[i] < 0:
        n_negative_before = n_negative_before + 1
    if savings_after[i] < 0:
        n_negative_after = n_negative_after + 1
    sum_saving_before = sum_saving_before + savings_before[i]
    sum_income_before = sum_income_before + income_numbers[i]
    sum_saving_after = sum_saving_after + savings_after[i]
    sum_income_after = sum_income_after + incomes_after[i]

pooled_before = sum_saving_before / sum_income_before
pooled_after = sum_saving_after / sum_income_after


def mean_of_defined(shares):
    """Return (mean of the shares that are not None, number included); (None, 0) if none."""
    total = 0
    count = 0
    for share in shares:
        if share is not None:
            total = total + share
            count = count + 1
    if count == 0:
        return None, 0
    return total / count, count


mean_before, count_before = mean_of_defined(shares_before)
mean_after, count_after = mean_of_defined(shares_after)

# Same households in both periods: those whose share is defined before support.
same_before = []
same_after = []
for i in range(n_households):
    if shares_before[i] is not None and shares_after[i] is not None:
        same_before.append(shares_before[i])
        same_after.append(shares_after[i])
mean_same_before, count_same = mean_of_defined(same_before)
mean_same_after, _ = mean_of_defined(same_after)

print(f"Total transfer: EUR {total_transfer:.2f} per month (all {n_households} households)")
print(f"Households with a positive transfer: {n_positive_transfer} of {n_households}")
print(f"Households with negative saving before support: {n_negative_before} of {n_households}")
print(f"Households with negative saving after support:  {n_negative_after} of {n_households}")
print(f"Pooled saving share before: {sum_saving_before:.2f} / {sum_income_before:.2f} = {pooled_before:.2%}")
print(f"Pooled saving share after:  {sum_saving_after:.2f} / {sum_income_after:.2f} = {pooled_after:.2%}")
print(f"Mean of defined household shares before: {mean_before:.2%} ({count_before} of {n_households} households)")
print(f"Mean of defined household shares after:  {mean_after:.2%} ({count_after} of {n_households} households)")
print(f"Same {count_same} households (share defined before): "
      f"mean before {mean_same_before:.2%}, mean after {mean_same_after:.2%}")

Total transfer: EUR 600.00 per month (all 8 households)
Households with a positive transfer: 5 of 8
Households with negative saving before support: 6 of 8
Households with negative saving after support:  2 of 8
Pooled saving share before: -400.00 / 8400.00 = -4.76%
Pooled saving share after:  200.00 / 9000.00 = 2.22%
Mean of defined household shares before: -6.19% (6 of 8 households)
Mean of defined household shares after:  -7.03% (7 of 8 households)
Same 6 households (share defined before): mean before -6.19%, mean after 0.13%


**Interpretation (≤ 120 words).** The pooled share rises from −4.76% to 2.22%: transfers of EUR 600 change total saving from −400 to +200 while total income rises from 8,400 to 9,000. The mean of defined household shares seems to fall, from −6.19% (6 households) to −7.03% (7 households). This is not a fall for the same households. A02 has zero income before support, so its share is undefined; after support its income is EUR 100 and its share is −50%, so it enters only the "after" mean. For the same 6 households, the mean rises from −6.19% to 0.13%. A comparison over different sets mixes a change in membership with a change in shares. Pooled shares weight by income; means weight households equally.

### 3(d) Inline function (lambda)

In [10]:
target_share = 0.10
defined_before = []
for share in shares_before:
    if share is not None:
        defined_before.append(share)

nearest_first = sorted(defined_before, key=lambda share: abs(share - target_share))

print("Defined pre-policy shares, nearest to 10% first:")
print(nearest_first)
for share in nearest_first:
    print(f"  {share:.2%}")
print("Original result list (unchanged, with None):", shares_before)

Defined pre-policy shares, nearest to 10% first:
[0.2222222222222222, -0.041666666666666664, -0.06666666666666667, -0.08333333333333333, -0.1875, -0.21428571428571427]
  22.22%
  -4.17%
  -6.67%
  -8.33%
  -18.75%
  -21.43%
Original result list (unchanged, with None): [-0.041666666666666664, None, -0.06666666666666667, -0.21428571428571427, -0.08333333333333333, -0.1875, None, 0.2222222222222222]


`sorted` calls the lambda once for each element of `defined_before`. The lambda receives one share (a float, for example −0.0417) and returns its distance from the target, `abs(share - 0.10)`. `sorted` uses these distances only as sort keys (smallest first) and returns the original shares with their signs, not the distances. `sorted` makes a new list, so `defined_before` and `shares_before` (with its `None`) do not change.

### 3(e) Counterfactual: A01 income 1200 → 1202

In [11]:
counterfactual_incomes = income_numbers.copy()        # independent copy
a01 = household_ids.index("A01")
counterfactual_incomes[a01] = 1202.0

c = consumptions[a01]
n = household_sizes[a01]
t_base = transfer(income_numbers[a01], c, n)
t_cf = transfer(counterfactual_incomes[a01], c, n)
y_base = income_numbers[a01] + t_base
y_cf = counterfactual_incomes[a01] + t_cf
s_base = y_base - c
s_cf = y_cf - c

print(f"{'A01':<26}{'I = 1200':>10}{'I = 1202':>10}{'Change':>10}")
print(f"{'Transfer':<26}{t_base:>10.2f}{t_cf:>10.2f}{t_cf - t_base:>10.2f}")
print(f"{'Income after support':<26}{y_base:>10.2f}{y_cf:>10.2f}{y_cf - y_base:>10.2f}")
print(f"{'Saving after support':<26}{s_base:>10.2f}{s_cf:>10.2f}{s_cf - s_base:>10.2f}")

# Verify that the original inputs did not change.
print("Original income_numbers[A01]:", income_numbers[a01])
print("income_text unchanged:", income_text == original_income_text)

A01                         I = 1200  I = 1202    Change
Transfer                       50.00      0.00    -50.00
Income after support         1250.00   1202.00    -48.00
Saving after support            0.00    -48.00    -48.00
Original income_numbers[A01]: 1200.0
income_text unchanged: True


**Interpretation (≤ 80 words).** No. The rule has a notch at EUR 600 per person. At income 1200 and size 2, A01 has exactly EUR 600 per person, so it is eligible and gets EUR 50; income after support is 1250. At 1202, it has EUR 601 per person, so the transfer falls to 0 and income after support is 1202. EUR 2 more original income gives EUR 48 less income after support, and saving after support falls from 0 to −48.

## Question 4. Audit an agent-written function

### 4(a) Four defects

1. **Accumulator reset (line 7).** `total = 0` is inside the loop, so each included share replaces the earlier sum. The function returns last share / count, not the arithmetic mean of the defined shares.
2. **Zero shares excluded (line 6).** `share != 0` drops households with a share of exactly 0. The contract says valid zero shares are included in the mean (and in the count).
3. **Silent truncation (line 4).** `zip` stops at the shorter list. Unequal lengths give a number from the matched pairs. The contract requires `ValueError`.
4. **Division by zero (line 10).** With empty inputs, or when no share is defined (all incomes zero), `count` is 0 and `total / count` raises `ZeroDivisionError`. The contract requires `None`.

The function does not modify its inputs, so that part of the contract is met.

In [12]:
def proposed_mean(incomes, consumptions):
    total = 0
    count = 0
    for income, consumption in zip(incomes, consumptions):
        share = saving_share(income, consumption)
        if share is not None and share != 0:
            total = 0
            total = total + share
            count = count + 1
    return total / count

### 4(b) Repaired function

In [13]:
def mean_defined_share(incomes, consumptions):
    """Arithmetic mean of defined saving shares (zero and negative shares included).

    Raises ValueError for unequal lengths. Returns None when no share is defined,
    including empty inputs. Does not modify its inputs.
    """
    if len(incomes) != len(consumptions):          # defect 3: check before the loop
        raise ValueError("incomes and consumptions must have equal lengths")
    total = 0                                       # defect 1: set once, before the loop
    count = 0
    for i in range(len(incomes)):
        share = saving_share(incomes[i], consumptions[i])
        if share is not None:                       # defect 2: keep zero shares
            total = total + share
            count = count + 1
    if count == 0:                                  # defect 4: no defined share
        return None
    return total / count

### 4(c) Diagnostic inputs

| Case | `incomes`, `consumptions` | Shares | Expected (contract) | Proposal gives | Defect exposed |
|---|---|---|---|---|---|
| 1 | `[1000, 1000]`, `[800, 900]` | 0.2, 0.1 | `0.15` | `0.05` (0.1 / 2) | 1, reset of `total` |
| 2 | `[1000, 1000]`, `[1000, 800]` | 0.0, 0.2 | `0.10` | `0.20` (zero share dropped) | 2, zero excluded |
| 3 | `[1000, 500]`, `[800]` | 0.2 only | `ValueError` | `0.2` (second income ignored) | 3, `zip` truncation |
| 4 | `[]`, `[]` | none | `None` | `ZeroDivisionError` | 4, division by zero |

Case 1 has only non-zero shares and equal lengths, so only the reset is visible. Case 2 includes only one non-zero share, so the reset has no effect and only the zero exclusion is visible. Case 3 has one matched pair, so only truncation is visible. Case 4 has no pair, so only the division is visible (`[0], [0]`, an all-undefined input, gives the same result).

Calls that run without an exception:

In [14]:
print("Case 1: proposed", proposed_mean([1000, 1000], [800, 900]),
      "| repaired", mean_defined_share([1000, 1000], [800, 900]))
print("Case 2: proposed", proposed_mean([1000, 1000], [1000, 800]),
      "| repaired", mean_defined_share([1000, 1000], [1000, 800]))
print("Case 3: proposed", proposed_mean([1000, 500], [800]))
print("Case 4: repaired", mean_defined_share([], []))
print("All undefined: repaired", mean_defined_share([0, 0], [0, 10]))

Case 1: proposed 0.05 | repaired 0.15000000000000002
Case 2: proposed 0.2 | repaired 0.1
Case 3: proposed 0.2
Case 4: repaired None
All undefined: repaired None


Deliberately failing calls (run in scratch cells; observed errors copied here, so the notebook runs without unhandled exceptions):

```python
>>> mean_defined_share([1000, 500], [800])      # case 3, repaired
ValueError: incomes and consumptions must have equal lengths

>>> proposed_mean([], [])                       # case 4, proposal
ZeroDivisionError: division by zero

>>> proposed_mean([0], [0])                     # all shares undefined, proposal
ZeroDivisionError: division by zero

>>> saving_share(-100, 50)                      # invalid amount, Question 2
ValueError: income and consumption must be non-negative

>>> transfer(1000, 500, 0)                      # invalid size, Question 2
ValueError: size must be a positive integer
```

## Question 5. Evidence

### 5(a) Assertions

In [15]:
# 1. Ordinary saving
assert math.isclose(saving_share(1000, 800), 0.2)
# 2. Negative saving is a valid negative share
assert math.isclose(saving_share(1200, 1250), -50 / 1200)
# 3. Zero income gives an undefined share
assert saving_share(0, 150) is None
# 4. Eligibility boundary: exactly EUR 600 per person is eligible; EUR 601 is not
assert transfer(1200, 1250, 2) == 50
assert transfer(1202, 1250, 2) == 0
# 5. Transfer cap: shortfall 150 is capped at 100 * 1
assert transfer(0, 150, 1) == 100
# 6. No shortfall gives zero (a number, not None)
assert transfer(1800, 1400, 3) == 0
assert transfer(0, 0, 2) == 0
# 7. All-undefined shares (and empty input) give None
assert mean_defined_share([0, 0], [0, 10]) is None
assert mean_defined_share([], []) is None
# 8. Counterfactual preserved the original inputs
assert income_numbers[a01] == 1200
assert counterfactual_incomes[a01] == 1202
assert income_text == original_income_text
# 9. Zero shares count in the mean
assert math.isclose(mean_defined_share([1000, 1000], [1000, 800]), 0.1)
# 10. Post-policy share reuses transfer: A01 has Y = 1250 = C
assert math.isclose(post_policy_share(1200, 1250, 2), 0.0)
# 11. Report totals agree with hand calculation
assert total_transfer == 600
assert math.isclose(pooled_before, -400 / 8400)
assert math.isclose(pooled_after, 200 / 9000)
# 12. Supplied lists are unchanged
assert household_ids == original_ids
assert consumptions == original_consumptions
assert household_sizes == original_sizes

print("All assertions passed.")

All assertions passed.


Plausible mistakes that four assertions detect:

- **Assertion 4 (boundary):** detects `income / size >= 600` instead of `> 600`, which would wrongly refuse support at exactly EUR 600 per person.
- **Assertion 5 (cap):** detects a missing cap, or a cap of EUR 100 per household instead of per person, which would pay the full shortfall.
- **Assertion 3 (zero income):** detects code that returns 0 (or divides by zero) for zero income instead of `None`.
- **Assertion 8 (counterfactual):** detects `counterfactual_incomes = income_numbers` without `.copy()`, which would also change A01's original income.

### 5(b) Reproducibility
This notebook was run with **Restart Kernel and Run All Cells**; every cell runs in order without manual input or unhandled exceptions. The first prediction for Question 1 is kept above.